# Evaluate LViTImproved on QaTa-COV19 Test Set

Notebook n?y ch?y inference tr?n `QaTa-COV19/Test_Folder` cho 2 checkpoint ?? train:

- `tqc0103/lvit-improved-qatacovid19-1-0`: m? h?nh 100% nh?n.
- `tqc0103/lvit-improved-qatacovid19-0-5`: m? h?nh 50% nh?n.

K?t qu? ch?nh c?n ??a v?o report l? `mean_test_dice` v? `mean_test_iou` trong file `test_summary.csv`.

Tr??c khi ch?y, v?o `Runtime > Change runtime type > T4 GPU` ho?c GPU t??ng ???ng.


In [ ]:
# Cell 1: Check GPU
!nvidia-smi


## 1. Clone ho?c mount repo

N?u repo c?a b?n ?? upload v?o `/content/LViT_improved` th? b? qua cell clone. N?u ch?a c? repo tr?n Colab, ch?nh `REPO_URL` r?i ch?y cell b?n d??i.


In [ ]:
# Cell 2: Prepare repo path
from pathlib import Path
import os

REPO_ROOT = Path('/content/LViT_improved')

# N?u b?n mu?n clone t? GitHub, b? comment 2 d?ng d??i v? s?a REPO_URL.
# REPO_URL = 'https://github.com/<your-user>/LViT_improved.git'
# !git clone {REPO_URL} {REPO_ROOT}

if not REPO_ROOT.exists():
    raise FileNotFoundError('Ch?a th?y /content/LViT_improved. H?y upload repo ho?c clone repo tr??c.')

os.chdir(REPO_ROOT)
print('Repo root:', Path.cwd())


## 2. C?i th? vi?n v? c?u h?nh Kaggle

Upload file `kaggle.json` v?o Colab, v? d? `/content/kaggle.json`. Kh?ng paste token v?o notebook.


In [ ]:
# Cell 3: Install dependencies
!pip install -q kaggle==1.6.17 transformers sentencepiece protobuf openpyxl opencv-python-headless pandas tqdm


In [ ]:
# Cell 4: Setup Kaggle credentials
from pathlib import Path
import shutil

kaggle_dir = Path('/root/.kaggle')
kaggle_dir.mkdir(parents=True, exist_ok=True)

source_token = Path('/content/kaggle.json')
target_token = kaggle_dir / 'kaggle.json'

if source_token.exists() and not target_token.exists():
    shutil.copy2(source_token, target_token)

if not target_token.exists():
    raise FileNotFoundError('Kh?ng th?y kaggle.json. H?y upload file v?o /content/kaggle.json r?i ch?y l?i cell n?y.')

target_token.chmod(0o600)
print('Kaggle token ready:', target_token)


## 3. T?i dataset v? kernel outputs

Dataset d?ng ??ng b?n trong `run_config.json`: `tqc0103/qata-covid19`. Kernel outputs ch?a `best_model.pt` cho 2 thi?t l?p.


In [ ]:
# Cell 5: Download dataset and trained kernel outputs
from pathlib import Path

WORK_ROOT = Path('/content/lvit_qatacov19_eval')
DATASET_DIR = WORK_ROOT / 'dataset'
DATASET_ROOT = DATASET_DIR / 'QaTa-Covid19'
KERNEL_OUTPUT_ROOT = WORK_ROOT / 'kaggle_outputs'
RESULT_ROOT = WORK_ROOT / 'results'

WORK_ROOT.mkdir(parents=True, exist_ok=True)
DATASET_DIR.mkdir(parents=True, exist_ok=True)
KERNEL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

if not (DATASET_ROOT / 'Test_Folder' / 'Test_text.xlsx').exists():
    !kaggle datasets download -d tqc0103/qata-covid19 -p {DATASET_DIR} --unzip
else:
    print('Dataset already exists:', DATASET_ROOT)

runs = {
    'lvit-improved-qatacovid19-100pct': {
        'kernel': 'tqc0103/lvit-improved-qatacovid19-1-0',
        'subdir': 'qatacov19_100pct',
    },
    'lvit-improved-qatacovid19-050pct': {
        'kernel': 'tqc0103/lvit-improved-qatacovid19-0-5',
        'subdir': 'qatacov19_050pct',
    },
}

for run_name, spec in runs.items():
    out_dir = KERNEL_OUTPUT_ROOT / run_name
    ckpt = out_dir / spec['subdir'] / 'best_model.pt'
    if ckpt.exists():
        print('Kernel output already exists:', ckpt)
        continue
    out_dir.mkdir(parents=True, exist_ok=True)
    !kaggle kernels output {spec['kernel']} -p {out_dir}


## 4. Import code v? ??nh ngh?a metric

Cell n?y d?ng class `LViTImproved` v? loader c? s?n trong repo.


In [ ]:
# Cell 6: Imports and metrics
import csv
import json
import sys
from pathlib import Path

import torch
import pandas as pd
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

sys.path.insert(0, str(REPO_ROOT))

from Load_Dataset import ImageToImage2D, ValGenerator
from nets.LViT_improved import LViTImproved
from text_encoder import attribute_vector_size, build_cache_metadata
from utils import read_text

TEXT_MODEL_NAME = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
MAX_TEXT_UNITS = 10
IMAGE_SIZE = 224
THRESHOLD = 0.5
BATCH_SIZE = 16
NUM_WORKERS = 2
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

def dice_score(target, prediction, eps=1e-6):
    target = target.float().reshape(target.shape[0], -1)
    prediction = prediction.float().reshape(prediction.shape[0], -1)
    intersection = (target * prediction).sum(dim=1)
    denominator = target.sum(dim=1) + prediction.sum(dim=1)
    return (2.0 * intersection + eps) / (denominator + eps)

def iou_score(target, prediction, eps=1e-6):
    target = target.float().reshape(target.shape[0], -1)
    prediction = prediction.float().reshape(prediction.shape[0], -1)
    intersection = (target * prediction).sum(dim=1)
    union = target.sum(dim=1) + prediction.sum(dim=1) - intersection
    return (intersection + eps) / (union + eps)


In [ ]:
# Cell 7: Build test loader
def build_test_loader(result_dir: Path):
    test_text = read_text(str(DATASET_ROOT / 'Test_Folder' / 'Test_text.xlsx'))
    cache_metadata = build_cache_metadata(model_name=TEXT_MODEL_NAME, max_units=MAX_TEXT_UNITS)
    test_dataset = ImageToImage2D(
        str(DATASET_ROOT / 'Test_Folder'),
        'QaTa-COV19',
        test_text,
        ValGenerator(output_size=[IMAGE_SIZE, IMAGE_SIZE]),
        image_size=IMAGE_SIZE,
        cache_dir=str(result_dir / 'text_cache' / 'Test_Folder'),
        text_model_name=TEXT_MODEL_NAME,
        local_files_only=False,
        cache_metadata=cache_metadata,
        max_text_units=MAX_TEXT_UNITS,
    )
    loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    return loader

def load_model(checkpoint_path: Path):
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    config = checkpoint.get('config', {})
    model = LViTImproved(
        n_channels=3,
        n_classes=1,
        img_size=IMAGE_SIZE,
        text_dim=int(config.get('text_dim', 768)),
        attribute_dim=attribute_vector_size(),
    )
    model.load_state_dict(checkpoint['model_state'])
    model.to(DEVICE)
    model.eval()
    return model, checkpoint


## 5. Ch?y inference test cho 2 checkpoint

Cell n?y l? ph?n ch?nh. Sau khi ch?y xong, xem b?ng `summary_df`.


In [ ]:
# Cell 8: Evaluate one checkpoint
def evaluate_one(run_name: str, spec: dict):
    result_dir = RESULT_ROOT / run_name
    result_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = KERNEL_OUTPUT_ROOT / run_name / spec['subdir'] / 'best_model.pt'
    if not checkpoint_path.exists():
        raise FileNotFoundError(checkpoint_path)

    print(f'\nEvaluating {run_name}')
    print('Checkpoint:', checkpoint_path)
    loader = build_test_loader(result_dir)
    model, checkpoint = load_model(checkpoint_path)

    rows = []
    dice_values = []
    iou_values = []

    with torch.inference_mode():
        for sample, names in tqdm(loader, desc=run_name):
            image = sample['image'].to(DEVICE, non_blocking=True)
            label = sample['label'].to(DEVICE, non_blocking=True)
            text = sample['text'].to(DEVICE, non_blocking=True)
            attributes = sample.get('attributes')
            if attributes is not None:
                attributes = attributes.to(DEVICE, non_blocking=True)

            logits = model(image, text_tokens=text, structured_attributes=attributes)
            prediction = (torch.sigmoid(logits) >= THRESHOLD).float()
            target = (label > 0).float()
            if target.ndim == prediction.ndim - 1:
                target = target.unsqueeze(1)

            batch_dice = dice_score(target, prediction).detach().cpu().tolist()
            batch_iou = iou_score(target, prediction).detach().cpu().tolist()
            dice_values.extend(batch_dice)
            iou_values.extend(batch_iou)
            rows.extend({'image': name, 'dice': d, 'iou': i} for name, d, i in zip(names, batch_dice, batch_iou))

    summary = {
        'run': run_name,
        'checkpoint_epoch': int(checkpoint.get('epoch', -1)),
        'checkpoint_best_val_dice': float(checkpoint.get('best_val_dice', 0.0)),
        'num_test_samples': len(rows),
        'mean_test_dice': float(sum(dice_values) / max(len(dice_values), 1)),
        'mean_test_iou': float(sum(iou_values) / max(len(iou_values), 1)),
        'checkpoint': str(checkpoint_path),
    }

    with (result_dir / 'test_metrics.json').open('w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    pd.DataFrame(rows).to_csv(result_dir / 'test_per_sample.csv', index=False)
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    return summary


In [ ]:
# Cell 9: Run all evaluations and save summary
summaries = []
for run_name, spec in runs.items():
    summaries.append(evaluate_one(run_name, spec))

summary_df = pd.DataFrame(summaries)
summary_path = WORK_ROOT / 'test_summary.csv'
summary_df.to_csv(summary_path, index=False)
summary_df


## 6. T?i k?t qu? v? m?y

Sau khi ch?y xong, t?i `test_summary.csv` v? ho?c copy b?ng `summary_df` v?o report.


In [ ]:
# Cell 10: Download summary CSV
from google.colab import files
print('Summary path:', summary_path)
files.download(str(summary_path))
